# LeetCode 2894 — Divisible and Non-divisible Sums Difference bằng Python

Link đề: https://leetcode.com/problems/divisible-and-non-divisible-sums-difference/ · Độ khó: **Easy** · Chủ đề: Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def differenceOfSums(self, n: int, m: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong các số nguyên từ `1` đến `n`, tính `num1` = **tổng các số không chia hết cho `m`**, `num2` = **tổng các số chia hết cho `m`**, rồi trả về `num1 - num2`.
2. Nắm phép **chia lấy dư `%`** để kiểm tra "chia hết", và `range(start, stop, step)` để **nhảy cách đều**.
3. Đi từ cách **vòng lặp duyệt từng số** → **một lần duyệt cộng/trừ theo dấu** → **`sum(range(m, n + 1, m))` cho các bội của `m`** → **công thức toán `O(1)`** dựa trên tổng cấp số cộng.
4. Hiểu **tổng `1 + 2 + ... + n = n(n+1)/2`** (mẹo của Gauss) và vì sao `num1 - num2 = tổng_tất_cả - 2·num2`.
5. Tránh các lỗi kinh điển: bỏ sót số `n` trong `range`, đảo dấu `num2 - num1`, dùng `/` (ra số thực) thay vì `//`, quên nhân `m` trong tổng các bội, xử lý sai khi `m > n`.

## 2. Đề bài & ví dụ

Cho hai số nguyên dương `n` và `m`. Xét các số nguyên từ `1` đến `n` (gồm cả `1` và `n`). Định nghĩa:

- `num1`: **tổng** các số trong đoạn `[1, n]` **không chia hết** cho `m`;
- `num2`: **tổng** các số trong đoạn `[1, n]` **chia hết** cho `m`.

Hãy **trả về `num1 - num2`**.

**Ví dụ 1**

- Input: `n = 10`, `m = 3`
- Output: `19`
- Giải thích: các số chia hết cho 3 trong `[1, 10]` là `3, 6, 9`, tổng `num2 = 18`; các số còn lại `1, 2, 4, 5, 7, 8, 10`, tổng `num1 = 37`. Kết quả `37 - 18 = 19`.

**Ví dụ 2**

- Input: `n = 5`, `m = 6`
- Output: `15`
- Giải thích: không có số nào trong `[1, 5]` chia hết cho 6, nên `num2 = 0`, `num1 = 1 + 2 + 3 + 4 + 5 = 15`.

**Ví dụ 3**

- Input: `n = 5`, `m = 1`
- Output: `-15`
- Giải thích: mọi số đều chia hết cho 1, nên `num1 = 0`, `num2 = 15`, kết quả `0 - 15 = -15`.

**Ràng buộc:** `1 <= n, m <= 1000`.

In [1]:
from typing import List
import random

# Dữ liệu ví dụ của đề: (n, m, đáp án)
examples = [(10, 3, 19), (5, 6, 15), (5, 1, -15)]
for n, m, expected in examples:
    print(f"n={n}, m={m} -> đáp án kỳ vọng {expected}")

n=10, m=3 -> đáp án kỳ vọng 19
n=5, m=6 -> đáp án kỳ vọng 15
n=5, m=1 -> đáp án kỳ vọng -15


## 3. Kiến thức nền

### 3.1. Phép chia lấy dư `%` và "chia hết"

`a % m` là **số dư** khi chia `a` cho `m`. Một số `a` **chia hết cho `m`** khi và chỉ khi **số dư bằng 0**, tức `a % m == 0`. Ví dụ `9 % 3 == 0` (chia hết), `10 % 3 == 1` (không chia hết).

In [2]:
print(9 % 3)          # 0  -> 9 chia hết cho 3
print(10 % 3)         # 1  -> 10 không chia hết cho 3
print(5 % 6)          # 5  -> số nhỏ hơn số chia: dư chính nó
print(7 % 1)          # 0  -> mọi số đều chia hết cho 1
print([x for x in range(1, 11) if x % 3 == 0])     # [3, 6, 9]

0
1
5
0
[3, 6, 9]


### 3.2. `range(start, stop, step)`

`range(1, 11)` sinh `1, 2, ..., 10` (**không gồm** `stop`; muốn tới `n` phải viết `n + 1`). Thêm tham số thứ ba `step` để **nhảy cách đều**: `range(3, 11, 3)` sinh `3, 6, 9`. Đây chính là dãy các **bội của 3** không vượt quá 10.

In [3]:
print(list(range(1, 11)))          # [1, 2, ..., 10]
print(list(range(3, 11, 3)))       # [3, 6, 9]: bội của 3 trong [1, 10]
print(list(range(6, 6, 6)))        # []: rỗng vì bắt đầu đã bằng stop
n, m = 5, 6
print(list(range(m, n + 1, m)))    # []: m > n nên không có bội nào

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
[3, 6, 9]
[]
[]


### 3.3. `sum(...)` và biểu thức sinh (generator expression)

`sum(iterable)` cộng mọi phần tử. Ta có thể truyền **biểu thức sinh** `x for x in ... if điều_kiện`, nó **tạo từng phần tử khi cần** (không dựng cả danh sách), nên tiết kiệm bộ nhớ.

In [4]:
print(sum([1, 2, 3, 4]))                                   # 10
print(sum(x for x in range(1, 11) if x % 3 != 0))          # 37: tổng các số KHÔNG chia hết cho 3
print(sum(x for x in range(1, 11) if x % 3 == 0))          # 18: tổng các số chia hết cho 3

10
37
18


### 3.4. Biểu thức điều kiện `a if điều_kiện else b`

Python cho phép chọn giá trị ngay trong một biểu thức: `giá_trị_1 if điều_kiện else giá_trị_2`. Ta sẽ dùng để **cộng `x` nếu không chia hết** và **trừ `x` (cộng `-x`) nếu chia hết**, gộp hai tổng làm một.

In [5]:
x, m = 6, 3
print(x if x % m != 0 else -x)       # -6: chia hết nên lấy -x
x = 7
print(x if x % m != 0 else -x)       # 7:  không chia hết nên lấy x
print(sum(x if x % 3 != 0 else -x for x in range(1, 11)))   # 19 = 37 - 18

-6
7
19


### 3.5. Tổng `1 + 2 + ... + n = n(n+1)/2` (mẹo của Gauss)

Ghép số đầu với số cuối: `1 + n`, `2 + (n-1)`, ... mỗi cặp đều bằng `n + 1`. Có `n/2` cặp, nên tổng là `n(n+1)/2`. Ví dụ `1 + 2 + ... + 10 = 10·11/2 = 55`. Công thức này đúng cả khi `n` lẻ (vì `n(n+1)` luôn chẵn, chia 2 ra số nguyên).

In [6]:
n = 10
print(sum(range(1, n + 1)))          # 55: cộng thủ công
print(n * (n + 1) // 2)              # 55: công thức
print(all(sum(range(1, k + 1)) == k * (k + 1) // 2 for k in range(0, 200)))   # đúng với mọi k thử

55
55
True


### 3.6. Phép chia nguyên `//` và khác biệt với `/`

`a // b` cho **thương nguyên** (làm tròn xuống), kết quả là `int`. `a / b` luôn ra **số thực** (`float`), dù chia hết. Với bài này cần kết quả **số nguyên**, nên khi chia `n(n+1)/2` hãy dùng `//`.

In [7]:
print(10 // 3, 10 / 3)               # 3  3.3333333333333335
print(10 // 5, 10 / 5)               # 2  2.0  <- float dù chia hết
print(1000 * 1001 // 2)              # 500500: int
print(1000 * 1001 / 2)               # 500500.0: float (không nên dùng cho đáp án)

3 3.3333333333333335
2 2.0
500500
500500.0


### 3.7. Có bao nhiêu bội của `m` trong `[1, n]`? `k = n // m`

Các bội của `m` không vượt quá `n` là `m, 2m, 3m, ..., km` với `k` lớn nhất thỏa `k·m <= n`, tức **`k = n // m`**. Ví dụ `n = 10`, `m = 3`: `k = 10 // 3 = 3` (đó là `3, 6, 9`). Nếu `m > n` thì `k = 0` (không có bội nào).

In [8]:
for n, m in [(10, 3), (5, 6), (5, 1), (12, 4)]:
    k = n // m
    print(f"n={n}, m={m}: k={k}, các bội =", [m * i for i in range(1, k + 1)])

n=10, m=3: k=3, các bội = [3, 6, 9]
n=5, m=6: k=0, các bội = []
n=5, m=1: k=5, các bội = [1, 2, 3, 4, 5]
n=12, m=4: k=3, các bội = [4, 8, 12]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy viết các số từ `1` đến `n = 10` rồi **tô màu**: số chia hết cho `m = 3` tô đỏ, các số còn lại tô xanh.

| số `x` | `x % 3` | chia hết? | cộng vào |
|---|---|---|---|
| 1 | 1 | không | `num1` |
| 2 | 2 | không | `num1` |
| 3 | 0 | **có** | `num2` |
| 4 | 1 | không | `num1` |
| 5 | 2 | không | `num1` |
| 6 | 0 | **có** | `num2` |
| 7 | 1 | không | `num1` |
| 8 | 2 | không | `num1` |
| 9 | 0 | **có** | `num2` |
| 10 | 1 | không | `num1` |

`num1 = 1 + 2 + 4 + 5 + 7 + 8 + 10 = 37`, `num2 = 3 + 6 + 9 = 18`, đáp án `37 - 18 = 19`. Đây là thuật toán: **mỗi số rơi vào đúng một nhóm** (chia hết hoặc không), cộng vào tổng nhóm ấy rồi lấy hiệu.

## 5. Cách 1 — Brute Force / baseline: duyệt từng số, hai biến tổng

**Ý tưởng.** Đặt `num1 = 0`, `num2 = 0`. Duyệt `x` từ `1` đến `n`: nếu `x % m == 0` thì `num2 += x`, ngược lại `num1 += x`. Cuối cùng trả `num1 - num2`. Đây là mô phỏng trực tiếp đề bài.

In [9]:
def difference_bruteforce(n: int, m: int) -> int:
    num1 = 0                      # tổng các số KHÔNG chia hết cho m
    num2 = 0                      # tổng các số chia hết cho m
    for x in range(1, n + 1):     # nhớ n + 1 để lấy được cả n
        if x % m == 0:
            num2 += x
        else:
            num1 += x
    return num1 - num2


for n, m, expected in examples:
    print(n, m, "->", difference_bruteforce(n, m), "(kỳ vọng", expected, ")")

10 3 -> 19 (kỳ vọng 19 )
5 6 -> 15 (kỳ vọng 15 )
5 1 -> -15 (kỳ vọng -15 )


In [10]:
# Theo dõi hai tổng cho n = 10, m = 3
n, m = 10, 3
num1 = num2 = 0
for x in range(1, n + 1):
    if x % m == 0:
        num2 += x
    else:
        num1 += x
print("num1 =", num1, "| num2 =", num2, "| num1 - num2 =", num1 - num2)

num1 = 37 | num2 = 18 | num1 - num2 = 19


**Độ phức tạp (Cách 1).** Vòng lặp chạy `n` lần, mỗi lần `O(1)`: **Time `O(n)`**; chỉ dùng vài biến số: **Space `O(1)`**.

**Nút thắt.** Với `n <= 1000` thì quá nhanh, nhưng nếu `n` lên tới `10^12` vòng lặp sẽ không chạy nổi. Ta tự hỏi: có tính được **không cần duyệt** từng số không?

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát giúp bỏ vòng lặp:

1. **Mỗi số thuộc đúng một nhóm:** `num1 + num2 = 1 + 2 + ... + n = n(n+1)/2` (gọi là `total`). Suy ra `num1 = total - num2`, nên

   `num1 - num2 = (total - num2) - num2 = total - 2·num2`.

   Chỉ cần tính **`num2`** (tổng các bội của `m`).

2. **Các bội của `m` tạo thành cấp số cộng:** `m, 2m, ..., km` với `k = n // m`. Tổng của chúng là

   `num2 = m(1 + 2 + ... + k) = m · k(k+1)/2`.

Vậy **đáp án = `n(n+1)/2 - 2 · m · k(k+1)/2`** với `k = n // m`, tính trong `O(1)`.

In [11]:
# Kiểm tra từng bước của công thức với n = 10, m = 3
n, m = 10, 3
total = n * (n + 1) // 2           # 55 = num1 + num2
k = n // m                         # 3 bội: 3, 6, 9
num2 = m * k * (k + 1) // 2        # 3 * (1 + 2 + 3) = 18
num1 = total - num2                # 37
print("total =", total, "| k =", k, "| num2 =", num2, "| num1 =", num1)
print("đáp án =", num1 - num2, "=", total - 2 * num2)      # 19

total = 55 | k = 3 | num2 = 18 | num1 = 37
đáp án = 19 = 19


## 7. Các cách tiếp theo

### Cách 2 — Một lần duyệt, cộng hoặc trừ theo dấu

**Giải thích.** Thay vì hai biến, ta gom thành **một tổng có dấu**: số không chia hết thì **cộng** (`+x`), số chia hết thì **trừ** (`-x`). Tổng cuối đúng là `num1 - num2`. Dùng biểu thức sinh và `sum`.

In [12]:
def difference_signed(n: int, m: int) -> int:
    return sum(x if x % m != 0 else -x for x in range(1, n + 1))


for n, m, expected in examples:
    print(n, m, "->", difference_signed(n, m), "(kỳ vọng", expected, ")")

10 3 -> 19 (kỳ vọng 19 )
5 6 -> 15 (kỳ vọng 15 )
5 1 -> -15 (kỳ vọng -15 )


In [13]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def differenceOfSums(self, n: int, m: int) -> int:
        return sum(x if x % m else -x for x in range(1, n + 1))

**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`** (biểu thức sinh không dựng danh sách). Ghi chú: `x % m` là `0` (được coi là "sai") khi chia hết, nên `x if x % m else -x` nghĩa là "nếu có dư thì `x`, nếu không thì `-x`".

### Cách 3 — `sum(range(m, n + 1, m))` cho các bội của `m`

**Giải thích.** `range(m, n + 1, m)` sinh trực tiếp các **bội của `m`** (không cần kiểm tra `%`). Vậy `num2 = sum(range(m, n + 1, m))`, `total = sum(range(1, n + 1))`, `num1 = total - num2`. Vẫn `O(n/m)` bước cho phần bội, nhanh hơn khi `m` lớn.

In [14]:
def difference_range_step(n: int, m: int) -> int:
    total = sum(range(1, n + 1))              # num1 + num2
    num2 = sum(range(m, n + 1, m))            # chỉ cộng các bội của m
    num1 = total - num2
    return num1 - num2


for n, m, expected in examples:
    print(n, m, "->", difference_range_step(n, m), "(kỳ vọng", expected, ")")

10 3 -> 19 (kỳ vọng 19 )
5 6 -> 15 (kỳ vọng 15 )
5 1 -> -15 (kỳ vọng -15 )


In [15]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def differenceOfSums(self, n: int, m: int) -> int:
        total = sum(range(1, n + 1))
        num2 = sum(range(m, n + 1, m))
        return total - 2 * num2

**Độ phức tạp Cách 3:** **Time `O(n)`** cho `total` (hoặc `O(1)` nếu dùng công thức Gauss) cộng `O(n/m)` cho các bội, **Space `O(1)`** (`range` là đối tượng "lười", không dựng danh sách).

### Cách 4 — Công thức toán `O(1)` (cách tối ưu)

**Giải thích.** Dùng hai công thức cấp số cộng ở mục 6: `total = n(n+1)/2` và `num2 = m·k(k+1)/2` với `k = n // m`. Trả `total - 2·num2`. **Không có vòng lặp**.

In [16]:
def difference_formula(n: int, m: int) -> int:
    total = n * (n + 1) // 2           # tổng 1..n
    k = n // m                         # số bội của m trong [1, n]
    num2 = m * k * (k + 1) // 2        # m + 2m + ... + km
    return total - 2 * num2


for n, m, expected in examples:
    print(n, m, "->", difference_formula(n, m), "(kỳ vọng", expected, ")")

10 3 -> 19 (kỳ vọng 19 )
5 6 -> 15 (kỳ vọng 15 )
5 1 -> -15 (kỳ vọng -15 )


In [17]:
# Code nộp LeetCode cho Cách 4
class Solution:
    def differenceOfSums(self, n: int, m: int) -> int:
        k = n // m                             # số bội của m trong [1, n]
        total = n * (n + 1) // 2               # num1 + num2
        multiples = m * k * (k + 1) // 2       # num2
        return total - 2 * multiples

**Độ phức tạp Cách 4:** **Time `O(1)`** (vài phép tính số học), **Space `O(1)`**. Đây là cách nên nộp và cách duy nhất vẫn chạy được khi `n` rất lớn.

> Ghi chú về kích thước số: Python có số nguyên "không giới hạn" nên không lo tràn số. Với ràng buộc của đề (`n <= 1000`) đáp án nhỏ hơn `500500`.

## 8. TRACE từng bước

Ta theo dõi (a) Cách 1 với `n = 10`, `m = 3` (từng số cộng vào nhóm nào); (b) Cách 4: từng bước của công thức.

In [18]:
def trace_bruteforce(n, m):
    num1 = num2 = 0
    print(f"n={n}, m={m}")
    for x in range(1, n + 1):
        r = x % m
        if r == 0:
            num2 += x
            action = f"chia hết -> num2 += {x}"
        else:
            num1 += x
            action = f"dư {r}     -> num1 += {x}"
        print(f"x={x:>2}  x%m={r}  {action:<24}  num1={num1:>3}  num2={num2:>3}")
    print(f"=> num1 - num2 = {num1} - {num2} = {num1 - num2}")
    return num1 - num2


assert trace_bruteforce(10, 3) == 19

n=10, m=3
x= 1  x%m=1  dư 1     -> num1 += 1     num1=  1  num2=  0
x= 2  x%m=2  dư 2     -> num1 += 2     num1=  3  num2=  0
x= 3  x%m=0  chia hết -> num2 += 3     num1=  3  num2=  3
x= 4  x%m=1  dư 1     -> num1 += 4     num1=  7  num2=  3
x= 5  x%m=2  dư 2     -> num1 += 5     num1= 12  num2=  3
x= 6  x%m=0  chia hết -> num2 += 6     num1= 12  num2=  9
x= 7  x%m=1  dư 1     -> num1 += 7     num1= 19  num2=  9
x= 8  x%m=2  dư 2     -> num1 += 8     num1= 27  num2=  9
x= 9  x%m=0  chia hết -> num2 += 9     num1= 27  num2= 18
x=10  x%m=1  dư 1     -> num1 += 10    num1= 37  num2= 18
=> num1 - num2 = 37 - 18 = 19


In [19]:
def trace_formula(n, m):
    print(f"n={n}, m={m}")
    total = n * (n + 1) // 2
    print(f"bước 1: total = n(n+1)/2 = {n}*{n + 1}/2 = {total}   (num1 + num2)")
    k = n // m
    print(f"bước 2: k = n // m = {n} // {m} = {k}   (các bội: {[m * i for i in range(1, k + 1)]})")
    num2 = m * k * (k + 1) // 2
    print(f"bước 3: num2 = m*k*(k+1)/2 = {m}*{k}*{k + 1}/2 = {num2}")
    num1 = total - num2
    print(f"bước 4: num1 = total - num2 = {total} - {num2} = {num1}")
    ans = total - 2 * num2
    print(f"bước 5: đáp án = total - 2*num2 = {total} - {2 * num2} = {ans}   (cũng bằng num1 - num2 = {num1 - num2})")
    return ans


assert trace_formula(10, 3) == 19
print()
assert trace_formula(5, 6) == 15       # k = 0: không có bội nào

n=10, m=3
bước 1: total = n(n+1)/2 = 10*11/2 = 55   (num1 + num2)
bước 2: k = n // m = 10 // 3 = 3   (các bội: [3, 6, 9])
bước 3: num2 = m*k*(k+1)/2 = 3*3*4/2 = 18
bước 4: num1 = total - num2 = 55 - 18 = 37
bước 5: đáp án = total - 2*num2 = 55 - 36 = 19   (cũng bằng num1 - num2 = 19)

n=5, m=6
bước 1: total = n(n+1)/2 = 5*6/2 = 15   (num1 + num2)
bước 2: k = n // m = 5 // 6 = 0   (các bội: [])
bước 3: num2 = m*k*(k+1)/2 = 6*0*1/2 = 0
bước 4: num1 = total - num2 = 15 - 0 = 15
bước 5: đáp án = total - 2*num2 = 15 - 0 = 15   (cũng bằng num1 - num2 = 15)


Bảng tóm tắt (Cách 4, `n = 10`, `m = 3`):

| bước | biểu thức | giá trị |
|---|---|---|
| 1 | `total = n(n+1)/2` | `55` |
| 2 | `k = n // m` | `3` |
| 3 | `num2 = m·k(k+1)/2` | `18` |
| 4 | `num1 = total - num2` | `37` |
| 5 | `total - 2·num2` | `19` |

## 9. Code nộp LeetCode

Bản nộp dùng công thức `O(1)` (Cách 4). Đúng class `Solution`, đúng method `differenceOfSums(self, n: int, m: int) -> int`.

In [20]:
class Solution:
    def differenceOfSums(self, n: int, m: int) -> int:
        k = n // m                             # số bội của m trong [1, n]
        total = n * (n + 1) // 2               # num1 + num2
        multiples = m * k * (k + 1) // 2       # num2
        return total - 2 * multiples


sol = Solution()
for n, m, expected in examples:
    print(n, m, "->", sol.differenceOfSums(n, m))

10 3 -> 19
5 6 -> 15
5 1 -> -15


## 10. Vì sao thuật toán đúng

- **Phân hoạch:** mỗi số `x` trong `[1, n]` hoặc chia hết cho `m` hoặc không, **không cả hai**. Do đó `num1 + num2 = 1 + 2 + ... + n = n(n+1)/2`.
- **Biến đổi hiệu:** từ `num1 = total - num2` suy ra `num1 - num2 = total - 2·num2`.
- **Các bội của `m`:** số `x` chia hết cho `m` khi `x = m·i` với `i` nguyên dương; điều kiện `x <= n` tương đương `i <= n // m = k`. Nên tập các số chia hết là `{m, 2m, ..., km}` (rỗng nếu `k = 0`).
- **Tổng các bội:** `m + 2m + ... + km = m·(1 + 2 + ... + k) = m·k(k+1)/2`. Công thức đúng cả khi `k = 0` (cho `0`).
- **Số nguyên:** `n(n+1)` và `k(k+1)` đều chẵn nên phép chia `// 2` chính xác (không mất phần lẻ).
- **Tính chất kiểm được:** công thức trùng khớp với duyệt trực tiếp với mọi cặp `(n, m)`; ta kiểm tra hết miền nhỏ ở phần test.

## 11. Phân tích Time / Space Complexity

- **Cách 1 / Cách 2:** Time `O(n)`, Space `O(1)`.
- **Cách 3:** Time `O(n)` nếu cộng `1..n` bằng `sum(range)`; phần các bội là `O(n/m)`. Space `O(1)`.
- **Cách 4 (công thức):** Time **`O(1)`** (số phép tính cố định), Space **`O(1)`**.
- Với ràng buộc `n, m <= 1000` mọi cách đều nhanh. Cách 4 mới là cách mở rộng được tới `n` cực lớn; còn Cách 1 là cách dễ kiểm tra nhất, nên ta dùng nó làm "chuẩn" để đối chiếu.

## 12. Bộ test

Ta kiểm tra tất cả cách bằng `assert`, với ví dụ của đề, biên, và đối chiếu toàn bộ miền nhỏ rồi ngẫu nhiên.

In [21]:
sol = Solution()
impls = {
    "brute": difference_bruteforce,
    "signed": difference_signed,
    "range_step": difference_range_step,
    "formula": difference_formula,
    "submit": sol.differenceOfSums,
}

for n, m, expected in examples:
    for key, fn in impls.items():
        assert fn(n, m) == expected, (key, n, m, fn(n, m), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [22]:
def check_all(n, m, expected, label):
    for key, fn in impls.items():
        assert fn(n, m) == expected, (label, key, n, m, fn(n, m), expected)
    print("PASS", label, f"(n={n}, m={m}) ->", expected)


check_all(1, 1, -1, "nhỏ nhất: 1 chia hết cho 1")
check_all(1, 2, 1, "n = 1, m > n: không có bội")
check_all(10, 10, 55 - 2 * 10, "m = n: chỉ n chia hết")
check_all(10, 11, 55, "m = n + 1: không có bội")
check_all(10, 1, -55, "m = 1: mọi số chia hết -> -(tổng)")
check_all(10, 2, (1 + 3 + 5 + 7 + 9) - (2 + 4 + 6 + 8 + 10), "m = 2: lẻ trừ chẵn")
check_all(1000, 1, -500500, "n lớn nhất, m = 1")
check_all(1000, 1000, 500500 - 2 * 1000, "n = m = 1000")
check_all(1000, 3, 500500 - 2 * 3 * 333 * 334 // 2, "n = 1000, m = 3")

PASS nhỏ nhất: 1 chia hết cho 1 (n=1, m=1) -> -1
PASS n = 1, m > n: không có bội (n=1, m=2) -> 1
PASS m = n: chỉ n chia hết (n=10, m=10) -> 35
PASS m = n + 1: không có bội (n=10, m=11) -> 55
PASS m = 1: mọi số chia hết -> -(tổng) (n=10, m=1) -> -55
PASS m = 2: lẻ trừ chẵn (n=10, m=2) -> -5
PASS n lớn nhất, m = 1 (n=1000, m=1) -> -500500
PASS n = m = 1000 (n=1000, m=1000) -> 498500
PASS n = 1000, m = 3 (n=1000, m=3) -> 166834


### 12.2. Tính chất và phản ví dụ

In [23]:
# Tính chất 1: m = 1 -> luôn là -n(n+1)/2
for n in range(1, 200):
    assert sol.differenceOfSums(n, 1) == -(n * (n + 1) // 2)
print("PASS m = 1 cho -n(n+1)/2")

# Tính chất 2: m > n -> luôn là n(n+1)/2
for n in range(1, 100):
    for m in range(n + 1, n + 20):
        assert sol.differenceOfSums(n, m) == n * (n + 1) // 2
print("PASS m > n cho n(n+1)/2")

# Tính chất 3: num1 + num2 = n(n+1)/2 và đáp án = total - 2*num2
for n, m in [(10, 3), (37, 5), (100, 7)]:
    num2 = sum(x for x in range(1, n + 1) if x % m == 0)
    num1 = sum(x for x in range(1, n + 1) if x % m != 0)
    assert num1 + num2 == n * (n + 1) // 2
    assert sol.differenceOfSums(n, m) == num1 - num2 == n * (n + 1) // 2 - 2 * num2
print("PASS num1 + num2 = tổng, đáp án = tổng - 2*num2")

# Kiểu trả về là int
assert isinstance(sol.differenceOfSums(10, 3), int)
print("PASS kiểu trả về là int")

# Phản ví dụ 1: hiệu bị đảo dấu (num2 - num1)
assert (18 - 37) != sol.differenceOfSums(10, 3) and (18 - 37) == -19
print("PASS phản ví dụ: num2 - num1 cho -19, không phải 19")

# Phản ví dụ 2: range(1, n) thiếu số n
wrong = sum(x if x % 3 else -x for x in range(1, 10))      # thiếu số 10
assert wrong == 19 - 10
print("PASS phản ví dụ: bỏ sót n cho", wrong, "thay vì 19")

PASS m = 1 cho -n(n+1)/2
PASS m > n cho n(n+1)/2
PASS num1 + num2 = tổng, đáp án = tổng - 2*num2
PASS kiểu trả về là int
PASS phản ví dụ: num2 - num1 cho -19, không phải 19
PASS phản ví dụ: bỏ sót n cho 9 thay vì 19


### 12.3. Đối chiếu toàn bộ miền nhỏ và ngẫu nhiên (có seed)

In [24]:
# Toàn bộ miền 1..60 x 1..60
for n in range(1, 61):
    for m in range(1, 61):
        ref = difference_bruteforce(n, m)
        for key, fn in impls.items():
            assert fn(n, m) == ref, (key, n, m)
print("PASS đối chiếu mọi (n, m) với 1 <= n, m <= 60")

# Ngẫu nhiên trong miền đề: 1..1000
random.seed(2894)
for _ in range(2000):
    n, m = random.randint(1, 1000), random.randint(1, 1000)
    ref = difference_bruteforce(n, m)
    for key, fn in impls.items():
        assert fn(n, m) == ref, (key, n, m)
print("PASS 2000 cặp (n, m) ngẫu nhiên - mọi cách khớp nhau")

PASS đối chiếu mọi (n, m) với 1 <= n, m <= 60


PASS 2000 cặp (n, m) ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `n + 1` trong `range` (bỏ sót số `n`)

In [25]:
def sai_off_by_one(n, m):
    return sum(x if x % m else -x for x in range(1, n))      # range(1, n) dừng ở n - 1


print("sai  :", sai_off_by_one(10, 3), "(đáp án đúng 19)")
print("đúng :", Solution().differenceOfSums(10, 3))

sai  : 9 (đáp án đúng 19)
đúng : 19


**Vì sao sai:** `range(1, n)` **không gồm** `n`. Đề yêu cầu đoạn `[1, n]` gồm cả `n`, nên phải viết `range(1, n + 1)`. Ở ví dụ trên bị thiếu số 10 (không chia hết cho 3), nên kết quả thấp hơn 10.

### Lỗi 2: Đảo dấu — trả `num2 - num1`

In [26]:
def sai_dao_dau(n, m):
    num1 = sum(x for x in range(1, n + 1) if x % m != 0)
    num2 = sum(x for x in range(1, n + 1) if x % m == 0)
    return num2 - num1


print("sai  :", sai_dao_dau(10, 3))                 # -19
print("đúng :", Solution().differenceOfSums(10, 3))  # 19

sai  : -19
đúng : 19


**Vì sao sai:** đề yêu cầu `num1 - num2` với `num1` là tổng số **không chia hết**. Nhầm vai trò hai tổng cho kết quả đổi dấu. Cách kiểm nhanh: ví dụ 3 (`n = 5`, `m = 1`) phải ra **âm**.

### Lỗi 3: Dùng `/` thay vì `//` trong công thức

In [27]:
def sai_chia_thuc(n, m):
    k = n // m
    return n * (n + 1) / 2 - 2 * (m * k * (k + 1) / 2)


r = sai_chia_thuc(10, 3)
print("sai  :", r, type(r).__name__)            # 19.0 float
print("đúng :", Solution().differenceOfSums(10, 3), type(Solution().differenceOfSums(10, 3)).__name__)

sai  : 19.0 float
đúng : 19 int


**Vì sao sai:** `/` luôn trả `float` (`19.0`). Giá trị bằng nhau nhưng **kiểu sai** (đề cần `int`), và với số rất lớn `float` mất chính xác (chỉ khoảng 15-16 chữ số). Hãy dùng `//` (an toàn vì tử số luôn chẵn).

### Lỗi 4: Quên nhân `m` khi tính tổng các bội

In [28]:
def sai_quen_m(n, m):
    k = n // m
    total = n * (n + 1) // 2
    num2 = k * (k + 1) // 2            # thiếu m: đây là tổng 1 + 2 + ... + k, không phải m + 2m + ... + km
    return total - 2 * num2


print("sai  :", sai_quen_m(10, 3), "(đáp án đúng 19)")
print("đúng :", Solution().differenceOfSums(10, 3))

sai  : 43 (đáp án đúng 19)
đúng : 19


**Vì sao sai:** các bội là `m, 2m, ..., km` chứ không phải `1, 2, ..., k`. Tổng là `m·(1 + ... + k)`. Kiểm tra bằng ví dụ: `3 + 6 + 9 = 18 = 3·6`, còn `1 + 2 + 3 = 6`.

### Lỗi 5: Xử lý `m > n` và `m = 1` sai (chia cho 0 / bội rỗng)

In [29]:
def sai_dung_range_cuoi(n, m):
    # tưởng rằng luôn có ít nhất một bội, nên cộng thẳng m
    num2 = m + sum(range(2 * m, n + 1, m))
    total = n * (n + 1) // 2
    return total - 2 * num2


print("sai  (n=5, m=6):", sai_dung_range_cuoi(5, 6), "(đáp án đúng 15)")
print("đúng (n=5, m=6):", Solution().differenceOfSums(5, 6))

sai  (n=5, m=6): 3 (đáp án đúng 15)
đúng (n=5, m=6): 15


**Vì sao sai:** khi `m > n` không có bội nào trong `[1, n]` (`k = 0`), nên `num2 = 0`; cộng cứng `m` làm kết quả lệch (ở đây thành `15 - 12 = 3`). Luôn để công thức/`range` tự cho kết quả rỗng khi không có bội; và nhớ trường hợp `m = 1` (mọi số đều là bội) cho đáp án âm `-n(n+1)/2`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "tổng các số chia hết / không chia hết cho `m` trong `[1, n]`" → nghĩ tới **`% m`**, **`range(m, n+1, m)`**, và **công thức cấp số cộng**.
- Công thức nhớ: **`k = n // m`**, **`total = n(n+1)//2`**, **`num2 = m·k(k+1)//2`**, **đáp án = `total - 2·num2`**.
- Khi hai nhóm **phủ kín và không giao nhau**, tính tổng của một nhóm rồi suy ra nhóm kia từ tổng tất cả.
- Luôn thử 3 trường hợp biên: `m = 1`, `m > n`, `m = n`.
- Dùng `//` cho phép chia nguyên; chỉ dùng `/` khi thực sự muốn số thực.
- Với `n` nhỏ cứ duyệt thẳng cho chắc, rồi dùng nó để kiểm chứng công thức.

## 15. Biến thể của bài

1. Tổng các số **chia hết** cho `m` hoặc cho `p`: dùng bao hàm - loại trừ (`m`, `p`, và `lcm(m, p)`).
2. Hiệu của tổng **bình phương** các số chia hết và không chia hết: cần công thức `1² + ... + k² = k(k+1)(2k+1)/6`.
3. Đếm **số lượng** số chia hết cho `m` trong `[a, b]`: `b // m - (a - 1) // m`.
4. Tổng các số trong `[a, b]` chia hết cho `m`: tổng bội đến `b` trừ tổng bội đến `a - 1`.
5. Chỉ cộng các số **chẵn** không chia hết cho `m`: duyệt `range(2, n + 1, 2)` kết hợp điều kiện `% m`.
6. Làm với `n` cực lớn (ví dụ `10^18`): công thức `O(1)` vẫn chạy, vòng lặp thì không.

Ta thử vài biến thể:

In [30]:
def sum_multiples_upto(n, m):
    k = n // m
    return m * k * (k + 1) // 2

n, m = 10, 3
print("3. đếm bội trong [4, 10]:", 10 // 3 - 3 // 3, "| duyệt:", sum(1 for x in range(4, 11) if x % 3 == 0))
print("4. tổng bội trong [4, 10]:", sum_multiples_upto(10, 3) - sum_multiples_upto(3, 3), "| duyệt:", sum(x for x in range(4, 11) if x % 3 == 0))
print("6. n = 10**18, m = 7:", Solution().differenceOfSums(10**18, 7))
from math import lcm
n, m, p = 30, 3, 5
exact = sum(x for x in range(1, n + 1) if x % m == 0 or x % p == 0)
formula = sum_multiples_upto(n, m) + sum_multiples_upto(n, p) - sum_multiples_upto(n, lcm(m, p))
print("1. chia hết cho 3 hoặc 5:", exact, formula)

3. đếm bội trong [4, 10]: 2 | duyệt: 2
4. tổng bội trong [4, 10]: 15 | duyệt: 15
6. n = 10**18, m = 7: 357142857142857142642857142857142858
1. chia hết cho 3 hoặc 5: 225 225


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt + hai biến tổng | mô phỏng đề, `if x % m == 0` | `O(n)` | `O(1)` | dễ hiểu, dùng để kiểm chứng |
| 2. Một lần duyệt, cộng/trừ theo dấu | `sum(x if x % m else -x ...)` | `O(n)` | `O(1)` | ngắn gọn, một dòng |
| 3. `sum(range(m, n+1, m))` | cộng thẳng các bội | `O(n)` hoặc `O(n/m)` | `O(1)` | không cần kiểm `%`, khi `m` lớn thì nhanh hơn |
| 4. Công thức cấp số cộng | `total - 2·m·k(k+1)/2` | `O(1)` | `O(1)` | **cách nộp bài**, scale tới `n` cực lớn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `17 % 5`, `list(range(4, 20, 4))`, `20 // 6`, `20 / 5`, `sum(range(1, 6))`.

**Bài 2 (tính tay).** Với `n = 12`, `m = 4`: liệt kê các bội của 4, tính `num2`, tính `total`, rồi suy ra `num1` và đáp án.

**Bài 3 (điền TODO).** Hoàn thành `difference_todo(n, m)` bằng công thức.

In [31]:
def difference_todo(n, m):
    # TODO 1: k = n // m
    # TODO 2: total = n * (n + 1) // 2
    # TODO 3: num2 = m * k * (k + 1) // 2
    # TODO 4: return total - 2 * num2
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `difference_loop2(n, m)` bằng vòng lặp duyệt `1..n` với hai biến `num1`, `num2` (Cách 1) mà không nhìn code mẫu. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `sum_not_divisible(n, m)` trả về chỉ `num1` bằng công thức `O(1)`. (b) Viết `diff_range(a, b, m)` trả về `num1 - num2` cho đoạn `[a, b]` (với `1 <= a <= b`) bằng công thức `O(1)`, dùng hàm "tổng bội tới `x`".

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `[4, 8, 12, 16]`, `3`, `4.0`, `15`. Bài 2: bội của 4 trong `[1, 12]` là `4, 8, 12`.
- **Mức 2 (vừa):** Bài 2: `num2 = 4 + 8 + 12 = 24`, `total = 78`, `num1 = 54`, đáp án `54 - 24 = 30`. Bài 3: bốn dòng đã ghi trong comment. Bài 4: `for x in range(1, n + 1)`, `if x % m == 0` thì cộng vào `num2`, ngược lại vào `num1`.
- **Mức 3 (sâu):** Bài 5(a): `num1 = total - num2` với `total = n(n+1)//2`, `num2 = m·k(k+1)//2`. Bài 5(b): đặt `S(x, m) = m·k(k+1)//2` với `k = x // m` và `T(x) = x(x+1)//2`; tổng bội trong `[a, b]` là `S(b) - S(a-1)`, tổng tất cả là `T(b) - T(a-1)`, đáp án `total_ab - 2·multiples_ab`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert 17 % 5 == 2
assert list(range(4, 20, 4)) == [4, 8, 12, 16]
assert 20 // 6 == 3
assert 20 / 5 == 4.0 and isinstance(20 / 5, float)
assert sum(range(1, 6)) == 15

# Bài 2
n, m = 12, 4
multiples = [x for x in range(1, n + 1) if x % m == 0]
assert multiples == [4, 8, 12]
num2 = sum(multiples)
total = n * (n + 1) // 2
assert (num2, total, total - num2, total - 2 * num2) == (24, 78, 54, 30)
assert Solution().differenceOfSums(12, 4) == 30

# Bài 3
def difference_todo_ans(n, m):
    k = n // m
    total = n * (n + 1) // 2
    num2 = m * k * (k + 1) // 2
    return total - 2 * num2

for n, m, expected in examples:
    assert difference_todo_ans(n, m) == expected

# Bài 4
def difference_loop2(n, m):
    num1 = 0
    num2 = 0
    for x in range(1, n + 1):
        if x % m == 0:
            num2 += x
        else:
            num1 += x
    return num1 - num2

for n, m, expected in examples:
    assert difference_loop2(n, m) == expected

# Bài 5(a)
def sum_not_divisible(n, m):
    k = n // m
    return n * (n + 1) // 2 - m * k * (k + 1) // 2

assert sum_not_divisible(10, 3) == 37 and sum_not_divisible(5, 1) == 0 and sum_not_divisible(5, 6) == 15

# Bài 5(b)
def diff_range(a, b, m):
    def multiples_upto(x):
        k = x // m
        return m * k * (k + 1) // 2

    def total_upto(x):
        return x * (x + 1) // 2

    total_ab = total_upto(b) - total_upto(a - 1)
    mult_ab = multiples_upto(b) - multiples_upto(a - 1)
    return total_ab - 2 * mult_ab

assert diff_range(1, 10, 3) == 19
for a in range(1, 20):
    for b in range(a, 25):
        for m in (1, 2, 3, 7):
            expected = sum(x if x % m else -x for x in range(a, b + 1))
            assert diff_range(a, b, m) == expected, (a, b, m)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hiệu hai tổng nhờ công thức cấp số cộng)

```python
class Solution:
    def differenceOfSums(self, n: int, m: int) -> int:
        k = n // m                          # số bội của m trong [1, n]
        total = n * (n + 1) // 2            # num1 + num2
        return total - 2 * (m * k * (k + 1) // 2)
```

**Checklist**

- [ ] `range(1, n + 1)` gồm cả `n` (nếu dùng vòng lặp)?
- [ ] Đúng thứ tự `num1 - num2` (không chia hết trừ chia hết), ví dụ `m = 1` ra số âm?
- [ ] Dùng `//` cho phép chia nguyên (kiểu trả về `int`), không dùng `/`?
- [ ] Tổng bội có nhân `m`: `m·k(k+1)//2`?
- [ ] Đã thử `m = 1`, `m > n`, `m = n`, `n = 1`?
- [ ] Đã đối chiếu công thức với vòng lặp trên một miền nhỏ?
- [ ] Skeleton đúng `class Solution` / `def differenceOfSums(self, n: int, m: int) -> int`?